# Differential Equations — Session 18
## Section 4.6: Variation of Parameters

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives


1. Motivate variation of parameters from first-order equations.
2. derive the two-equation system for $u_1',u_2'$.
3. apply the Wronskian formulas.
4. use integral-defined particular solutions.
5. compare variation of parameters with undetermined coefficients.
6. verify integral responses numerically.


> The theoretical sequence follows the supplied section, while all examples, diagrams, and simulations are original.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |\n|---:|---|\n| 0–15 min | Motivation from varying constants |\n| 15–40 min | Derivation and Wronskian formulas |\n| 40–60 min | Elementary-integral example |\n| 60–78 min | Nonelementary forcing |\n| 78–88 min | Numerical verification |\n| 88–90 min | Exit check |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE=True
except ImportError:
    WIDGETS_AVAILABLE=False

if os.environ.get('NB_VALIDATION_MODE') == '1':
    WIDGETS_AVAILABLE=False
plt.rcParams['figure.figsize']=(8,5)
plt.rcParams['axes.grid']=True
np.set_printoptions(precision=6,suppress=True)

def solve_second_order(f, span, y0, yp0, points=900, **kwargs):
    def rhs(x,z): return [z[1], f(x,z[0],z[1])]
    t=np.linspace(span[0],span[1],points)
    return solve_ivp(rhs,span,[y0,yp0],t_eval=t,**kwargs)

def wronskian_symbolic(funcs,x):
    return sp.simplify(sp.Matrix([[sp.diff(f,x,j) for f in funcs] for j in range(len(funcs))]).det())

print('Notebook ready.')
print('Interactive widgets available:',WIDGETS_AVAILABLE)

## Formal theory reference

For

$$
y''+P(x)y'+Q(x)y=f(x),
$$

let $y_1,y_2$ be a fundamental set and $W=y_1y_2'-y_1'y_2$. Seek

$$
y_p=u_1y_1+u_2y_2.
$$

Impose

$$
y_1u_1'+y_2u_2'=0,
$$

$$
y_1'u_1'+y_2'u_2'=f.
$$

Then

$$
u_1'=-\frac{y_2f}{W},\qquad u_2'=\frac{y_1f}{W}.
$$

The equation must first be divided by the leading coefficient. Constants of integration are omitted because they reproduce complementary terms.

### Classroom Checkpoint — Variation of Parameters Denominator

In the second-order variation-of-parameters formulas, which quantity appears in the denominator, and why must it be nonzero?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Example: $y''+y=e^x$

Use $y_1=\cos x$, $y_2=\sin x$, $W=1$.

In [ ]:
x=sp.symbols('x',real=True); f=sp.exp(x); y1=sp.cos(x); y2=sp.sin(x); W=1
u1=sp.integrate(-y2*f/W,x); u2=sp.integrate(y1*f/W,x); yp=sp.simplify(u1*y1+u2*y2)
display(u1); display(u2); display(sp.simplify(yp)); display(sp.simplify(sp.diff(yp,x,2)+yp-f))

## 2. Integral-defined response

For

$$
y''+y=\frac{1}{1+x^2},\qquad y(0)=y'(0)=0,
$$

variation of parameters yields the rest response

$$
y(x)=\int_0^x\sin(x-t)\frac{1}{1+t^2}\,dt.
$$

In [ ]:
def vop_integral(xvals):
    return np.array([quad(lambda t: np.sin(x-t)/(1+t*t),0,x)[0] for x in xvals])
xv=np.linspace(0,15,500); yv=vop_integral(xv)
sol=solve_ivp(lambda t,z:[z[1],1/(1+t*t)-z[0]],(0,15),[0,0],t_eval=xv,rtol=1e-9,atol=1e-11)
plt.plot(xv,yv,label='variation-of-parameters integral'); plt.plot(xv,sol.y[0],linestyle='--',label='solve_ivp'); plt.legend(); plt.show(); print(np.max(np.abs(yv-sol.y[0])))

## 3. Interactive forcing comparison

Variation of parameters works for forcing functions outside the undetermined-coefficients catalog.

In [ ]:
def forcing_response(kind='rational',T=20):
    if kind=='rational': fun=lambda t:1/(1+t*t)
    elif kind=='gaussian': fun=lambda t:np.exp(-t*t/4)
    else: fun=lambda t:np.log1p(t)
    x=np.linspace(0,T,500); y=np.array([quad(lambda s:np.sin(t-s)*fun(s),0,t)[0] for t in x])
    plt.plot(x,y,label='response'); plt.plot(x,[fun(t) for t in x],linestyle='--',label='forcing'); plt.legend(); plt.show()
if WIDGETS_AVAILABLE: interact(forcing_response,kind=Dropdown(options=['rational','gaussian','logarithm'],value='rational'),T=IntSlider(min=5,max=30,step=5,value=20))
else: forcing_response()

## 4. Method comparison

Undetermined coefficients is efficient but restricted. Variation of parameters is broadly applicable whenever a fundamental set and required integrals are available.

## Classroom Checkpoint — Exit Check

For $y''-y=f(x)$ with $y_1=e^x,y_2=e^{-x}$, compute $W$ and state $u_1',u_2'$.

> Pause here. Let students commit to an answer before running the next cell.